# Cycle Clean — 数据清洗与提取

本 notebook 仿照 cycle_anchor.ipynb，仅做**数据清洗**与**列提取**。

**清洗规则：**
1. 清洗 **cycle0**（排除从研究开始到第一次 Menstrual 之前的不完整周期）
2. 清洗 **周期天数 &lt; 6 天** 的周期
3. 清洗 **周期内天数不连续** 的周期：缺失 **1 天以上** 的周期剔除（只缺失 1 天的周期不剔除）

**注意：** 若某周期内存在 **LH 或 estrogen 缺失** 的行，该周期**不参与上述清洗**（即不做剔除，原样保留）。

## 1. 导入与路径

In [11]:
import pandas as pd
import numpy as np
import os

data_dir = '../mcphases-a-dataset-of-physiological-hormonal-and-self-reported-events-and-symptoms-for-menstrual-health-tracking-with-wearables-1.0.0'
output_dir = '../subdataset'
input_file = os.path.join(data_dir, 'hormones_and_selfreport.csv')
output_file = os.path.join(output_dir, 'cycle_clean_2.csv')

os.makedirs(output_dir, exist_ok=True)

## 2. 加载原始数据并提取指定列

提取列：`id`, `study_interval`, `day_in_study`, `phase`, `lh`, `estrogen`

In [12]:
df_original = pd.read_csv(input_file)
columns_to_extract = ['id', 'study_interval', 'day_in_study', 'phase', 'lh', 'estrogen']
df = df_original[columns_to_extract].copy()

print(f"Original shape: {df_original.shape}")
print(f"After extract shape: {df.shape}")
print(df.head())

Original shape: (5659, 22)
After extract shape: (5659, 6)
   id  study_interval  day_in_study       phase   lh  estrogen
0   1            2022             1  Follicular  2.9      94.2
1   1            2022             2  Follicular  1.2     226.3
2   1            2022             3  Follicular  3.5     276.8
3   1            2022             4   Fertility  1.8     322.1
4   1            2022             5   Fertility  4.6     244.9


## 3. 分组：大组 (id + study_interval) 与周期 (Cycle)

- **大组**：按 `id` + `study_interval` 分组  
- **周期**：从第一次出现 Menstrual 起算一个周期；研究开始到第一次 Menstrual 之前为 cycle0

In [13]:
def assign_cycle_ids(group_df):
    phases = group_df['phase'].values
    n = len(phases)
    cycle_ids = []
    current_cycle = 0
    left_menstrual = False
    for i in range(n):
        phase = phases[i]
        if i == 0:
            if phase == 'Menstrual':
                current_cycle = 1
                left_menstrual = False
            else:
                current_cycle = 0
                left_menstrual = True
        else:
            if phase != 'Menstrual':
                left_menstrual = True
            elif phase == 'Menstrual' and left_menstrual:
                current_cycle += 1
                left_menstrual = False
        cycle_ids.append(current_cycle)
    return cycle_ids

df['big_group'] = df['id'].astype(str) + '_' + df['study_interval'].astype(str)
df['small_group'] = -1
for big_group_name, big_group_df in df.groupby('big_group', sort=False):
    cycle_ids = assign_cycle_ids(big_group_df)
    df.loc[big_group_df.index, 'small_group'] = cycle_ids
df['small_group_key'] = df['big_group'] + '_cycle' + df['small_group'].astype(str)

print(f"Big groups: {df['big_group'].nunique()}, cycles: {df['small_group_key'].nunique()}")

Big groups: 62, cycles: 241


## 3.5 LH / estrogen 缺失值填补

- 用**上下两行的平均值**填补 lh、estrogen 的缺失（按周期内 `day_in_study` 排序后，取前一行与后一行的均值）。
- **若某周期内存在连续缺失 > 5 天**（即连续 6 行及以上为缺失），则该周期**整周期删除**（不参与后续填补与清洗）；并打印被删除的周期 key。

In [14]:
# Per cycle: if consecutive missing > 5 days skip fill; else fill with prev/next row mean
cycles_skip_fill = []  # cycles with consecutive missing > 5 days
df_filled = df.copy()

for sk, g in df.groupby('small_group_key', sort=False):
    g = g.sort_values('day_in_study')
    missing = g['lh'].isna() | g['estrogen'].isna()
    if not missing.any():
        continue
    # Consecutive missing length
    run = 0
    max_run = 0
    for m in missing:
        if m:
            run += 1
            max_run = max(max_run, run)
        else:
            run = 0
    if max_run > 5:
        cycles_skip_fill.append(sk)
        continue
    # Fill with prev/next row mean (from df_filled so consecutive gaps use already-filled values)
    idx = g.index.tolist()
    n = len(idx)
    for i in range(n):
        if pd.isna(g['lh'].iloc[i]):
            prev_lh = df_filled.loc[idx[i - 1], 'lh'] if i > 0 else np.nan
            next_lh = df_filled.loc[idx[i + 1], 'lh'] if i < n - 1 else np.nan
            if not pd.isna(prev_lh) and not pd.isna(next_lh):
                df_filled.loc[idx[i], 'lh'] = (float(prev_lh) + float(next_lh)) / 2
            elif not pd.isna(prev_lh):
                df_filled.loc[idx[i], 'lh'] = float(prev_lh)
            elif not pd.isna(next_lh):
                df_filled.loc[idx[i], 'lh'] = float(next_lh)
        if pd.isna(g['estrogen'].iloc[i]):
            prev_est = df_filled.loc[idx[i - 1], 'estrogen'] if i > 0 else np.nan
            next_est = df_filled.loc[idx[i + 1], 'estrogen'] if i < n - 1 else np.nan
            if not pd.isna(prev_est) and not pd.isna(next_est):
                df_filled.loc[idx[i], 'estrogen'] = (float(prev_est) + float(next_est)) / 2
            elif not pd.isna(prev_est):
                df_filled.loc[idx[i], 'estrogen'] = float(prev_est)
            elif not pd.isna(next_est):
                df_filled.loc[idx[i], 'estrogen'] = float(next_est)

df = df_filled
# Remove cycles with consecutive missing > 5 days (drop whole cycle from data)
df = df[~df['small_group_key'].isin(cycles_skip_fill)].copy().reset_index(drop=True)
print(f"Cycles with consecutive missing>5 days (removed): {len(cycles_skip_fill)}")
if cycles_skip_fill:
    for sk in cycles_skip_fill:
        print(sk)
else:
    print("(none)")
print(f"Rows after removal: {len(df)}")

Cycles with consecutive missing>5 days (removed): 6
8_2022_cycle2
8_2022_cycle3
22_2022_cycle3
23_2022_cycle2
42_2024_cycle4
49_2022_cycle2
Rows after removal: 5559


## 4. 清洗规则（仅对 LH/estrogen 无缺失的周期生效）

- 若某周期内**存在任意一行** lh 或 estrogen 缺失，该周期**不参与清洗**，整周期保留。  
- 仅对**周期内 lh、estrogen 均无缺失**的周期做以下剔除：  
  1. 剔除 **cycle0**  
  2. 剔除 **周期天数 &lt; 6 天** 的周期  
  3. 剔除 **周期内缺失天数 &gt; 1** 的周期（只缺 1 天保留）

In [15]:
# Per-cycle stats: has lh/estrogen missing, row count, day continuity
cycle_info = []
for sk, g in df.groupby('small_group_key', sort=False):
    parts = sk.rsplit('_cycle', 1)
    big_group, cycle_num = parts[0], int(parts[1])
    n_rows = len(g)
    has_missing = g['lh'].isna().any() or g['estrogen'].isna().any()
    days = sorted(g['day_in_study'].dropna().astype(int).tolist())
    if len(days) >= 2:
        expected = set(range(days[0], days[-1] + 1))
        actual = set(days)
        n_missing_days = len(expected - actual)
    else:
        n_missing_days = 0
    cycle_info.append({
        'small_group_key': sk,
        'big_group': big_group,
        'cycle_num': cycle_num,
        'n_rows': n_rows,
        'has_lh_estrogen_missing': has_missing,
        'n_missing_days': n_missing_days
    })
cycle_df = pd.DataFrame(cycle_info)
print("Cycle stats sample:")
print(cycle_df.head(12).to_string(index=False))

Cycle stats sample:
small_group_key big_group  cycle_num  n_rows  has_lh_estrogen_missing  n_missing_days
  1_2022_cycle0    1_2022          0      21                    False               0
  1_2022_cycle1    1_2022          1      30                    False               0
  1_2022_cycle2    1_2022          2      26                    False               0
  1_2022_cycle3    1_2022          3      13                    False               0
  2_2022_cycle0    2_2022          0      19                    False               0
  2_2022_cycle1    2_2022          1      31                    False               0
  2_2022_cycle2    2_2022          2      32                    False               0
  2_2022_cycle3    2_2022          3       8                    False               0
  3_2022_cycle0    3_2022          0       9                    False               0
  3_2022_cycle1    3_2022          1      33                    False               0
  3_2022_cycle2    3_2022         

In [16]:
# Decide whether to keep each cycle
# Cycles with LH/estrogen missing: do not apply cleaning, keep
# Cycles with no missing: drop cycle0, length<6, missing days>1
def should_keep(row):
    if row['has_lh_estrogen_missing']:
        return True
    if row['cycle_num'] == 0:
        return False
    if row['n_rows'] < 6:
        return False
    if row['n_missing_days'] > 1:
        return False
    return True

cycle_df['keep'] = cycle_df.apply(should_keep, axis=1)
kept_keys = set(cycle_df[cycle_df['keep']]['small_group_key'])
dropped = cycle_df[~cycle_df['keep']]
print(f"Cycles kept: {cycle_df['keep'].sum()}, cycles dropped: {(~cycle_df['keep']).sum()}")
print("\nDropped cycles (first 20):")
print(dropped[['small_group_key', 'cycle_num', 'n_rows', 'n_missing_days', 'has_lh_estrogen_missing']].head(20).to_string(index=False))

Cycles kept: 173, cycles dropped: 62

Dropped cycles (first 20):
small_group_key  cycle_num  n_rows  n_missing_days  has_lh_estrogen_missing
  1_2022_cycle0          0      21               0                    False
  2_2022_cycle0          0      19               0                    False
  3_2022_cycle0          0       9               0                    False
  6_2022_cycle0          0      18               0                    False
  7_2022_cycle4          4       5               0                    False
  8_2022_cycle0          0      24               0                    False
  9_2022_cycle0          0      19               0                    False
  9_2024_cycle0          0       2               0                    False
 10_2022_cycle0          0       1               0                    False
 10_2024_cycle0          0       7               0                    False
 10_2024_cycle3          3      16              14                    False
 11_2022_cycle0        

In [17]:
# Keep only rows belonging to cycles that pass the rules
df_clean = df[df['small_group_key'].isin(kept_keys)].copy()
df_clean = df_clean.sort_values(['id', 'study_interval', 'day_in_study']).reset_index(drop=True)
print(f"Rows after cleaning: {len(df_clean)}")
print(f"Cycles after cleaning: {df_clean['small_group_key'].nunique()}")

Rows after cleaning: 4825
Cycles after cleaning: 173


## 5. 排卵期概率（同 cycle_anchor.ipynb）

按 cycle_anchor 方式：月经结束日、基线 LH（月经后 1–4 天去极值平均）、LH/基线 ≥ 2.5 的 surge 段；取含周期最大 LH 的段，得到 onset_day / peak_day；对 τ~U(7,24) 边际化得 P(ovulation\|onset) 与 P(ovulation\|peak)，fused = 归一化 0.5·P(O\|onset)+0.5·P(O\|peak)。将三列写入输出表。

In [18]:
from scipy.stats import truncnorm
from scipy.integrate import quad

def find_menstruation_end_day(cycle_df):
    """day_in_study of the last Menstrual-phase day in the cycle."""
    menstrual_days = cycle_df[cycle_df['phase'] == 'Menstrual']
    if len(menstrual_days) == 0:
        return None
    return menstrual_days['day_in_study'].max()

def calculate_baseline(cycle_df, menstruation_end_day):
    """Mean LH on days 1–4 after menses end, after removing min/max."""
    if menstruation_end_day is None:
        baseline_days = cycle_df.head(4)
    else:
        start_day = menstruation_end_day + 1
        end_day = menstruation_end_day + 4
        baseline_days = cycle_df[
            (cycle_df['day_in_study'] >= start_day) &
            (cycle_df['day_in_study'] <= end_day)
        ]
    if baseline_days.empty:
        return None
    lh_values = baseline_days['lh'].dropna().values
    if lh_values.size == 0:
        return None
    if lh_values.size < 3:
        return float(np.mean(lh_values))
    lh_values = np.delete(lh_values, [np.argmax(lh_values), np.argmin(lh_values)])
    return float(np.mean(lh_values))

def find_surge_segments(cycle_df, baseline, menstruation_end_day):
    """Consecutive day segments where LH/baseline >= 2.5, (start_day, end_day); merge if gap <= 2 days."""
    if baseline is None or baseline == 0 or menstruation_end_day is None:
        return []
    start_check_day = menstruation_end_day + 5
    check_days = cycle_df[cycle_df['day_in_study'] >= start_check_day].copy()
    if len(check_days) == 0:
        return []
    check_days = check_days[check_days['lh'].notna()].copy()
    check_days['ratio'] = check_days['lh'] / baseline
    check_days = check_days.sort_values('day_in_study').reset_index(drop=True)
    surge_mask = check_days['ratio'] >= 2.5
    if surge_mask.sum() == 0:
        return []
    surge_segments = []
    in_surge, surge_start, prev_day = False, None, None
    for idx in range(len(check_days)):
        day = check_days.iloc[idx]['day_in_study']
        is_surge = surge_mask.iloc[idx]
        if is_surge:
            if not in_surge:
                surge_start = day
                in_surge = True
            elif prev_day is not None and day != prev_day + 1:
                surge_segments.append((surge_start, prev_day))
                surge_start = day
        else:
            if in_surge:
                surge_segments.append((surge_start, prev_day))
                in_surge = False
        prev_day = day
    if in_surge:
        surge_segments.append((surge_start, prev_day))
    merged = []
    for seg in surge_segments:
        if merged and seg[0] - merged[-1][1] <= 2:
            merged[-1] = (merged[-1][0], seg[1])
        else:
            merged.append(seg)
    return merged

# τ~U(7,24); Onset->ovulation TruncNorm(μ=36h,σ=6h,[22,47]h); Peak->ovulation TruncNorm(μ=12h,σ=4h,[8,20]h)
_TAU_LO, _TAU_HI = 7.0, 24.0
_TAU_WEIGHT = 1.0 / (_TAU_HI - _TAU_LO)
_OVULATION_ONSET_MU, _OVULATION_ONSET_SIGMA = 36.0, 6.0
_OVULATION_ONSET_LOW, _OVULATION_ONSET_HIGH = 22.0, 47.0
_onset_a = (_OVULATION_ONSET_LOW - _OVULATION_ONSET_MU) / _OVULATION_ONSET_SIGMA
_onset_b = (_OVULATION_ONSET_HIGH - _OVULATION_ONSET_MU) / _OVULATION_ONSET_SIGMA
_dist_onset = truncnorm(_onset_a, _onset_b, loc=_OVULATION_ONSET_MU, scale=_OVULATION_ONSET_SIGMA)
_OVULATION_PEAK_MU, _OVULATION_PEAK_SIGMA = 12.0, 4.0
_OVULATION_PEAK_LOW, _OVULATION_PEAK_HIGH = 8.0, 20.0
_peak_a = (_OVULATION_PEAK_LOW - _OVULATION_PEAK_MU) / _OVULATION_PEAK_SIGMA
_peak_b = (_OVULATION_PEAK_HIGH - _OVULATION_PEAK_MU) / _OVULATION_PEAK_SIGMA
_dist_peak = truncnorm(_peak_a, _peak_b, loc=_OVULATION_PEAK_MU, scale=_OVULATION_PEAK_SIGMA)

def _p_day_given_tau(tau, k, dist, low_support, high_support):
    start_h = max(0.0, 24.0 * k - tau)
    end_h = 24.0 * (k + 1) - tau
    if start_h >= end_h:
        return 0.0
    start_clip = max(low_support, start_h)
    end_clip = min(high_support, end_h)
    if start_clip >= end_clip:
        return 0.0
    return float(dist.cdf(end_clip) - dist.cdf(start_clip))

def prob_ovulation_given_onset(onset_day, day_in_study):
    k = day_in_study - onset_day
    if k < 0:
        return 0.0
    integral, _ = quad(
        lambda tau: _p_day_given_tau(tau, k, _dist_onset, _OVULATION_ONSET_LOW, _OVULATION_ONSET_HIGH),
        _TAU_LO, _TAU_HI, limit=50
    )
    return float(integral * _TAU_WEIGHT)

def prob_ovulation_given_peak(peak_day, day_in_study):
    k = day_in_study - peak_day
    if k < 0:
        return 0.0
    integral, _ = quad(
        lambda tau: _p_day_given_tau(tau, k, _dist_peak, _OVULATION_PEAK_LOW, _OVULATION_PEAK_HIGH),
        _TAU_LO, _TAU_HI, limit=50
    )
    return float(integral * _TAU_WEIGHT)

print("Ovulation probability helpers defined.")

Ovulation probability helpers defined.


In [19]:
# Use small_group_key as cycle_key
df_clean['cycle_key'] = df_clean['small_group_key'].copy()
df_clean['ovulation_day_method1'] = 0.0
df_clean['ovulation_day_method2'] = 0.0
df_clean['ovulation_prob_fused'] = 0.0

for cycle_key, cycle_df in df_clean.groupby('cycle_key', sort=False):
    cycle_df = cycle_df.sort_values('day_in_study').copy()
    menstruation_end_day = find_menstruation_end_day(cycle_df)
    baseline = calculate_baseline(cycle_df, menstruation_end_day)
    if baseline is None or baseline == 0:
        continue
    surge_segments = find_surge_segments(cycle_df, baseline, menstruation_end_day)
    if len(surge_segments) == 0:
        continue
    cycle_lh = cycle_df['lh'].dropna()
    max_lh_val = cycle_lh.max()
    max_lh_days = cycle_df.loc[cycle_df['lh'] == max_lh_val, 'day_in_study'].values
    max_lh_day = max_lh_days[0] if len(max_lh_days) > 0 else None
    selected_surge = None
    for seg in surge_segments:
        if max_lh_day is not None and seg[0] <= max_lh_day <= seg[1]:
            selected_surge = seg
            break
    if selected_surge is None:
        selected_surge = surge_segments[-1]
    surge_start_day, surge_end_day = selected_surge
    surge_data = cycle_df[
        (cycle_df['day_in_study'] >= surge_start_day) &
        (cycle_df['day_in_study'] <= surge_end_day)
    ].copy()
    surge_data = surge_data[surge_data['lh'].notna()]
    if len(surge_data) == 0:
        continue
    onset_day = surge_start_day
    peak_days_global = surge_data[surge_data['lh'] == surge_data['lh'].max()]['day_in_study'].values
    peak_day = int(min(peak_days_global))
    fused_by_day = {}
    for _, row in cycle_df.iterrows():
        d = row['day_in_study']
        p1 = prob_ovulation_given_onset(onset_day, d)
        p2 = prob_ovulation_given_peak(peak_day, d)
        df_clean.loc[(df_clean['cycle_key'] == cycle_key) & (df_clean['day_in_study'] == d), 'ovulation_day_method1'] = p1
        df_clean.loc[(df_clean['cycle_key'] == cycle_key) & (df_clean['day_in_study'] == d), 'ovulation_day_method2'] = p2
        fused_by_day[d] = 0.5 * p1 + 0.5 * p2
    total = sum(fused_by_day.values())
    if total > 0:
        for d, val in fused_by_day.items():
            df_clean.loc[(df_clean['cycle_key'] == cycle_key) & (df_clean['day_in_study'] == d), 'ovulation_prob_fused'] = val / total

df_clean['ovulation_day_method1'] = df_clean['ovulation_day_method1'].fillna(0.0).astype(float)
df_clean['ovulation_day_method2'] = df_clean['ovulation_day_method2'].fillna(0.0).astype(float)
df_clean['ovulation_prob_fused'] = df_clean['ovulation_prob_fused'].fillna(0.0).astype(float)
print(f"Ovulation prob labelled; non-zero method1: {(df_clean['ovulation_day_method1'] > 0).sum()} rows, fused: {(df_clean['ovulation_prob_fused'] > 0).sum()} rows")

Ovulation prob labelled; non-zero method1: 265 rows, fused: 360 rows


In [20]:
# Output columns include three ovulation prob columns; write to CSV
out_cols = ['id', 'study_interval', 'day_in_study', 'phase', 'lh', 'estrogen', 'big_group', 'small_group_key',
            'ovulation_day_method1', 'ovulation_day_method2', 'ovulation_prob_fused']
df_clean[out_cols].to_csv(output_file, index=False)
print(f"Saved: {output_file}")
print(f"Total rows: {len(df_clean)}, cycles: {df_clean['small_group_key'].nunique()}")
df_clean[out_cols].head(10)

Saved: ../subdataset/cycle_clean_2.csv
Total rows: 4825, cycles: 173


,id,study_interval,day_in_study,phase,lh,estrogen,big_group,small_group_key,ovulation_day_method1,ovulation_day_method2,ovulation_prob_fused
0,1,2022,22,Menstrual,2.7,123.9,1_2022,1_2022_cycle1,0.0,0.0,0.0
1,1,2022,23,Menstrual,4.0,140.2,1_2022,1_2022_cycle1,0.0,0.0,0.0
2,1,2022,24,Menstrual,2.8,152.7,1_2022,1_2022_cycle1,0.0,0.0,0.0
3,1,2022,25,Menstrual,2.8,147.7,1_2022,1_2022_cycle1,0.0,0.0,0.0
4,1,2022,26,Menstrual,1.6,69.6,1_2022,1_2022_cycle1,0.0,0.0,0.0
5,1,2022,27,Follicular,2.1,143.9,1_2022,1_2022_cycle1,0.0,0.0,0.0
6,1,2022,28,Follicular,1.7,70.4,1_2022,1_2022_cycle1,0.0,0.0,0.0
7,1,2022,29,Follicular,3.1,85.5,1_2022,1_2022_cycle1,0.0,0.0,0.0
8,1,2022,30,Follicular,1.8,75.2,1_2022,1_2022_cycle1,0.0,0.0,0.0
9,1,2022,31,Follicular,2.8,53.5,1_2022,1_2022_cycle1,0.0,0.0,0.0
